# Where do the errors come from?

Every model gets **the same blocks** from the PDF. So any difference between them
is the model, not the reading.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json
import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import load_method, micro_prf1

MODELS, EXTRACTOR, SAMPLES = ['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b'], 'pdfplumber', range(1, 11)
EXAMPLE = 5                     # document used for the example in step 2

pd.set_option('display.max_colwidth', 60)
INK = '#111111'
TINT = {'title': '#e3edfa', 'section.title': '#fce8df',
        'section.description': '#ddf3ec', 'question.text': '#ece3fa',
        'answer.text': '#eceef0'}


def tint(v):
    """One colour per label, so a run of one colour is visible at a glance."""
    return f'background-color: {TINT[v]}; color: {INK}' if v in TINT else ''


available = [m for m in MODELS
             if all(P.structured_path(P.make_tag(m, EXTRACTOR), n).exists()
                    for n in SAMPLES)]
print('models ready:', ', '.join(available))


models ready: llama3.1:8b, gemma4:e4b


## Step 1 — The puzzle

Same PDFs, same blocks, very different scores.


In [2]:
def label_runs(blocks):
    """How many times the label changes as you read down the document."""
    labs = [b.get('label') for b in blocks]
    return 1 + sum(1 for i in range(1, len(labs)) if labs[i] != labs[i - 1])


rows = []
for m in available:
    tag = P.make_tag(m, EXTRACTOR)
    blocks = changes = 0
    for n in SAMPLES:
        bl = json.loads(P.labeled_path(tag, n).read_text(encoding='utf-8'))
        blocks += len(bl)
        changes += label_runs(bl)
    rows.append({'model': m, 'blocks given': blocks,
                 'label changes': changes,
                 'changes per block': changes / blocks,
                 'f1': micro_prf1(load_method(tag, exclude=[])[1])['f1']})

df = pd.DataFrame(rows).set_index('model')
display(df.style
        .background_gradient(cmap='Reds', subset=['changes per block'])
        .background_gradient(cmap='Greens', subset=['f1'])
        .format({'changes per block': '{:.2f}', 'f1': '{:.3f}'}))


,blocks given,label changes,changes per block,f1
model,,,,
llama3.1:8b,729,461,0.63,0.298
gemma4:e4b,729,164,0.22,0.689


Every model was handed the **same number of blocks**. The model that changes its
label most often scores worst.

Step 2 shows why that matters so much.


## Step 2 — Why changing labels is expensive

pdfplumber reads **line by line**, so one paragraph arrives as several blocks.

The pipeline can put them back together — it joins neighbouring blocks that have
**the same label**. So:

```
answer  answer  answer  answer      ->  joined into 1  ->  correct
answer  question  answer  question  ->  stays as 4     ->  1 right, 3 wrong
```

A model that keeps the same label through a paragraph gets it rebuilt for free.
A model that flips back and forth leaves it in pieces, and every extra piece counts
as an error.

Here is that happening on a real document. **The text is identical** — only the
labels differ.


In [3]:
labeled = {m: json.loads(P.labeled_path(P.make_tag(m, EXTRACTOR), EXAMPLE)
                         .read_text(encoding='utf-8')) for m in available}
n_blocks = len(next(iter(labeled.values())))

# Show the stretch where the models disagree most.
if len(available) >= 2:
    a, b = available[0], available[1]
    diff = [i for i in range(n_blocks)
            if labeled[a][i].get('label') != labeled[b][i].get('label')]
    start = max(0, (diff[len(diff) // 2] if diff else 0) - 5)
else:
    start = 0
window = range(start, min(start + 12, n_blocks))

ex = pd.DataFrame([{**{m: labeled[m][i].get('label') for m in available},
                    'text': labeled[available[0]][i]['text']} for i in window],
                  index=[f'block {i}' for i in window])
display(ex.style.map(tint, subset=available))


,llama3.1:8b,gemma4:e4b,text
block 36,answer.text,question.text,correct attribution to original collectors. All methodological procedures for data quality control and
block 37,question.text,answer.text,aggregation will be described in the data documentation. This information will be provided in the form of
block 38,answer.text,question.text,"supplementary documentation for publications, in the README.txt file, and at the repository (record-"
block 39,answer.text,answer.text,"level). All sharable raw data and the processed data will be archived in Dryad, https://datadryad.org, the"
block 40,section.description,answer.text,"agnostic open access repository backed by UCSB and the California Digital Library (CDL), which"
block 41,question.text,answer.text,"implements the latest best practices in data curation, publication, citation, and archival. Dryad assigns"
block 42,answer.text,answer.text,Digital Object Identifiers (DOIs) to deposited datasets and makes data discoverable through such services
block 43,question.text,answer.text,"as the Thomson-Reuters Data Citation Index, Scopus, and Google Dataset Search. All data in Dryad is"
block 44,answer.text,answer.text,publicly accessible and released under a CC0 license. Data will be linked to publications and vice versa
block 45,question.text,answer.text,through DOIs.


Look down each model's column. **A single colour running down several rows** means
those blocks get joined into one item — one correct answer.

**Alternating colours** mean the same sentences stay split apart, and all but one of
the pieces is counted wrong.


## Step 3 — What this means

- **The PDF reader is not the problem.** It gives every model the same blocks, and
  loses no text. It just cuts a paragraph into more pieces than the annotation has.

- **Staying consistent is the skill being tested.** The pipeline repairs the cutting
  by itself, but only for a model that holds one label across a line break.

- **So a weak model looks far worse here than it really is at understanding text.**
  Most of its errors are correct text, correctly read, broken into pieces.

That is also why the line-merging step that used to run before labelling helped the
small model so much and the larger ones barely at all — it was doing this joining in
advance, for a model that could not do it itself.
